In [1]:
clean_up = True
%run stdPackages.ipynb
os.chdir(d['py'])
from modelData import *
from gmsPython import nestingTree
from multOutIO import AdjustIO, CreateIOTargets
from mWaste import EstimateTreatCosts, CalibFromPWTD, CalibToGR

# Create Model Data

For the "small" model, we still require the full model to get estimates of relative cost of waste management across $m$.

## 0. Preliminaries

*Define main settings:*

In [2]:
t0 = 2019
T  = t0 + 100 # set finite time horizon
name = 'vGRSIntRC' # Some global name that carries through all models/data to identify this version

*This creates a copy of the IO data from the ```IOdata``` project into the local data folder*

In [3]:
ioName = f'IO{t0}_WCGE_GRS'
ioNameF = f'IO{t0}_WCGE_GR'
shutil.copy(os.path.join(d['project'], 'IOdata','data','processedData', ioName),  os.path.join(d['data'], ioName));
shutil.copy(os.path.join(d['project'], 'IOdata','data','processedData', ioNameF), os.path.join(d['data'], ioNameF));
shutil.copy(os.path.join(d['project'], 'IOdata','data', 'wasteDataMapping.xlsx'),  os.path.join(d['data'], 'wasteDataMapping.xlsx'));

*Load data:*

In [4]:
db = GpyDB(os.path.join(d['data'], ioName), name = f'{name}{t0}_db', ws = d['work'])
db0 = db.copy() # create copy 
dbF = GpyDB(os.path.join(d['data'], ioNameF), ws = db.ws)

*Load data on waste part as well:*

In [5]:
wasteData = DbFromExcel.simpleLoad(os.path.join(d['data'], 'WasteDataFull.xlsx'))
wasteMaps = DbFromExcel.simpleLoad(os.path.join(d['data'], 'wasteDataMapping.xlsx'))

Read mappings:

In [6]:
dfMaps = pd.DataFrame(wasteMaps['s2025'].values).ffill()
dfMaps = pd.DataFrame(dfMaps.iloc[1:].values, columns = dfMaps.iloc[0]).astype(str)

*For relevant variables, add yearly index to data (this is currently all of them)*

In [7]:
def add_t0(k):
    k.index = stdSort(pd.MultiIndex.from_frame(k.index.to_frame(index=False).assign(t = t0)))
[add_t0(db(k)) for k in db.getTypes(['var','par'])];

*Add various time subsets that we'll rely on:*

In [8]:
addTimeToDB(t0, T, db)

*Specify waste index and waste incineration:*

In [9]:
db['n_waste'] = db('sWaste').rename('n')
db['n_wasteE'] = pd.Index(['WasteE'], name = 'n')

## 1. Material and waste data

### 1.1. Read and arrange data

*Read symbols:*

In [10]:
def readSymbol(wasteData, sheet, columns = ['i','m','s','o']):
    df = pd.DataFrame(wasteData[sheet].values)
    return pd.DataFrame(df.iloc[1:,:].values, columns = df.iloc[0]).set_index(columns)[sheet[0]]
wd = {s: readSymbol(wasteData, s) for s in ('qd','qs','vd','vs')}

*Adjust values to measure in mia. DKK (or scale of database; assumes that the materials data measures in 1000 DKK). Adjust quantities to be 100.000 ton (leaves quantities close to the ranges 0.1-10):*

In [11]:
[wd.__setitem__(k, wd[k]*1000/db._scale) for k in ('vd','vs')];
[wd.__setitem__(k, wd[k]/1e5) for k in ('qd','qs')];

*Update names:*

In [12]:
nameMap = {'Export': 'F', 'Import': 'F', 'Household': 'HH', 'Government': 'G'}
for k in wd:
    wd[k].index = wd[k].index.set_levels(wd[k].index.levels[0].map(lambda x: nameMap[x] if x in nameMap else x), level = 0)
    wd[k] = wd[k].rename_axis(index = {'s':'k','i':'s'}).droplevel('o')

*Check balances:*

In [13]:
df = pd.concat([wd['qd'].groupby(['m','k']).sum().rename('d'),
                wd['qs'].groupby(['m','k']).sum().rename('s')], axis = 1)
def testBalance(d, s, error = 1e-6):
    return all(abs(d.groupby(['m','k']).sum()-s.groupby(['m','k']).sum())<error)
assert all([testBalance(wd[f'{k}d'], wd[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

Create aggregate version:

In [14]:
m = pd.MultiIndex.from_frame(dfMaps[['s121','sGRS']], names = ['s','ss']).unique()
wdAgg = dict.fromkeys(wd)
for k in wd:
    wdAgg[k] = adjMultiIndex.applyMult(adj.rc_pd(wd[k], m), m).groupby(['ss','m','k']).sum().rename_axis(index = {'ss':'s'}).combine_first(
                adj.rc_pd(wd[k], ('not', m)))

### 1.2. Adhoc adjustments

#### A. Residual industry

For now, map residual industry to investment sector and treat as follows:
* Remove demand for waste materials. Increase domestic waste treatment instead.
* Map residual material supply to other industries with weights from size of capital stocks (we can only do this at the same industry level as the main database).
* Add residual demand for materials (non-waste) to industries that produce durables. 

In [15]:
wIdx = pd.Index(['W'], name = 'k')
wsIdx = db('sWaste')
rIdx = pd.Index(['Residual'], name = 's')

*Remove residual demand for waste materials. Add to waste treatment industries.*

In [16]:
wasteDemand = adj.rc_pd(wdAgg['qd'], ('and', [wIdx, rIdx]))
wasteTreatment = adj.rc_pd(wdAgg['qd'], wsIdx).add(wasteDemand.groupby(['m','k']).sum(), fill_value=0)
wdAgg['qd'] = wasteTreatment.combine_first(adj.rc_pd(wdAgg['qd'], ('not', wasteDemand)))

*Relative share of waste subindustries:*

In [17]:
weIdxFull = pd.Index(['383901'], name = 's') # sector identifier used to identify waste for incineration
wtIdxFull = pd.Index(['383902'], name = 's') # sector identifier used to identify waste for treatment.
wsIdxFull = weIdxFull.union(wtIdxFull)
wasteSizes = adj.rc_pd(wd['qd'], ('and', [wsIdxFull, wIdx]))
wasteSizes = wasteSizes/wasteSizes.groupby(['m']).sum()
wd['qd'] = wd['qd'].add((wasteSizes*wasteDemand.groupby(['m','k']).sum()).dropna(),fill_value=0)

*Map material supply to other industries with weights from relative capital stocks:*

In [18]:
vK = db('vD_dur').xs(t0).groupby('s').sum()
vK = vK/vK.sum()
for x in ('vs','qs'):
    resSupply = stdSort(adjMultiIndex.bc(wdAgg[x].xs('Residual'), vK) * vK)
    wdAgg[x] = adj.rc_pd(wdAgg[x], ('not', rIdx)).add(resSupply, fill_value=0)

*Map material demand to durable producers:*

In [19]:
vI = adj.rc_pd(db('vS'), db('s_i')).xs(t0).groupby('s').sum()
vI = vI/vI.sum() # this is simply 1 if we only have one type of durable
for x in ('vd','qd'):
    resDemand = stdSort(adjMultiIndex.bc(wdAgg[x].xs('Residual'), vI) * vI)
    wdAgg[x] = adj.rc_pd(wdAgg[x], ('not', rIdx)).add(resDemand, fill_value=0)

*Check balances again.*

In [20]:
assert all([testBalance(wdAgg[f'{k}d'], wdAgg[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

#### B. Household and government supply of (non-waste) materials

The current model structure does not include material supply from households and government. If these are here, we reduce exports appropriately:

In [21]:
sNoSupply = pd.Index(['HH','G'], name = 's') # what 's' does not supply anything (beyond waste)
kxW = pd.Index(['R','V'], name = 'k') # k types that are not waste
for x in ('q','v'):
    noSupply = adj.rc_pd(wdAgg[f'{x}s'], ('and', [sNoSupply, kxW]))
    wdAgg[f'{x}s'] = adj.rc_pd(wdAgg[f'{x}s'], ('not', noSupply))
    wdAgg[f'{x}d'] = wdAgg[f'{x}d'].sub(stdSort(adjMultiIndex.bc(noSupply.groupby(['m','k']).sum(), db('s_f'))),fill_value=0)

*Check balances again.*

In [22]:
assert all([testBalance(wdAgg[f'{k}d'], wdAgg[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

Similarly, the government does not seem to demand any services from the waste management sector in general. Thus, we remove the waste generated from this part and reduce exports appropriately:

In [23]:
sNoWaste = pd.Index(['G'], name = 's')
noWaste = adj.rc_pd(wdAgg['qs'], ('and', [sNoWaste, wIdx]))
wdAgg['qs'] = adj.rc_pd(wdAgg['qs'], ('not', noWaste))
wdAgg['qd'] = (adj.rc_pd(wdAgg['qd'], db('s_f')).sub(noWaste.groupby(['m','k']).sum(), fill_value=0)).combine_first(wdAgg['qd'])

#### C. Import-to-exports

Set import-to-export to x% of the *minimum* of exports and imports. This ensures non-negative imports/exports.

In [24]:
share = 0.80
for x in ('q','v'):
    reduction = pd.concat([adj.rc_pd(wdAgg[f'{x}d'], db('s_f')), adj.rc_pd(wdAgg[f'{x}s'], db('s_f'))], axis = 1).min(axis=1) * share
    wdAgg[f'{x}s'] = wdAgg[f'{x}s'].sub(reduction, fill_value=0)
    wdAgg[f'{x}d'] = wdAgg[f'{x}d'].sub(reduction, fill_value=0)

Check feasibility of waste treatment:

In [25]:
assert (wdAgg['qd'].xs(('Waste','W'), level = ('s','k'))-wdAgg['qs'].xs(('F','W'), level = ('s','k'))).min()>0, """ Import of waste exceeds total domestic treatment"""

*Adhoc adjustment: If exports exceed domestic supply of a material sort, we reduce imports and exports with the same amount to ensure this remains balanced:*

In [26]:
for k in ('v','q'):
    domesticSupply = adj.rc_pd(wdAgg[f'{k}s'], ('not', db('s_f'))).groupby(['m','k']).sum()
    diff = domesticSupply.sub(wdAgg[f'{k}d'].xs('F'),fill_value=0)
    wdAgg[f'{k}d'] = (adj.rc_pd(wdAgg[f'{k}d'], db('s_f'))+diff[diff<0]).dropna().combine_first(wdAgg[f'{k}d'])
    wdAgg[f'{k}s'] = (adj.rc_pd(wdAgg[f'{k}s'], db('s_f'))+diff[diff<0]).dropna().combine_first(wdAgg[f'{k}s'])
    wdAgg[f'{k}d'] = wdAgg[f'{k}d'][wdAgg[f'{k}d']!=0]
    wdAgg[f'{k}s'] = wdAgg[f'{k}s'][wdAgg[f'{k}s']!=0]

Update index levels for the following section:

In [27]:
for k in wdAgg:
    wdAgg[k].index = wdAgg[k].index.remove_unused_levels()

### 1.3. Target recycling rates

Get $E,W,A,D,R,\overline{R}$ from data:

In [28]:
E = adj.rc_pd(wd['qd'], ('and', [weIdxFull, wIdx])).groupby('m').sum() # waste for incineration 
W = adj.rc_pd(wdAgg['qd'], ('and', [wsIdx, wIdx])).groupby('m').sum() # total treated waste - this gives some problems with recycling efficiency > 100 %
A = wdAgg['qs'].xs(('Waste','R'),level=('s','k')) # recycled materials
dmin = 0 # share of materials that are landfill
D = dmin * W # landfill
R = W-E-D # waste going towards recycling
R̄ = W-D # Recyclable waste
# Load targets:
rates = pd.DataFrame(wasteData['targetRates'].values)
rates = pd.DataFrame(rates.iloc[1:,].values, columns = rates.iloc[0]).set_index('m')

Compute $R, W, E$ that is consistent with targets for $A/R, R/W$ in data:

In [29]:
Rb = A/(rates['A/R'].combine_first(A/R))
Wb = Rb /(rates['R/W'].combine_first(Rb/W))
Eb = Wb-Rb-D

Scale waste industry's demand for waste, scale all industries' supply of waste. In the disaggregated version, adjust waste for incineration and for recycling:

In [30]:
wdAgg['qd'] = (adj.rc_pd(wdAgg['qd'], ('and', [db('sWaste'), wIdx])) * Wb/W).combine_first(wdAgg['qd'])
wdAgg['qs'] = wdAgg['qs'] * wdAgg['qd'].groupby(['m','k']).sum()/wdAgg['qs'].groupby(['m','k']).sum()
wd_WE = adj.rc_pd(wd['qd'], ('and', [weIdxFull, wIdx])) * Eb/E # implied waste for incineration
wd_WT = adj.rc_pd(wd['qd'], ('and', [wtIdxFull, wIdx])) * (Wb-Eb)/(W-E) # implied waste for treatment 
wd['qd'] = wd_WE.combine_first(wd_WT).combine_first(wd['qd']) # update

*Old version*

Load target rates:

In [31]:
# rates = pd.DataFrame(wasteData['targetRates'].values)
# rates = pd.DataFrame(rates.iloc[1:,].values, columns = rates.iloc[0]).set_index('m')['A/R']
# rates = rates.combine_first(A/R) # if no target is provided, use rates from our data

Scale up all waste intake from domestic treatment firm to target this; scale up generation from all sources proportionally to source this. (this scales up waste for incineration and treatment proportionally).

In [32]:
# Rtarget = A/rates
# Radj = Rtarget/R
# wdAgg['qd'] = (adj.rc_pd(wdAgg['qd'], ('and', [db('sWaste'), wIdx])) * Radj).combine_first(wdAgg['qd'])
# wdAgg['qs'] = wdAgg['qs'] * wdAgg['qd'].groupby(['m','k']).sum()/wdAgg['qs'].groupby(['m','k']).sum()
# wd['qd'] = (adj.rc_pd(wd['qd'], ('and', [pd.Index(['383901','383902'], name = 's'), wIdx])) * Radj).combine_first(wd['qd'])

### 1.4. Remove small values

If demand/supply values are less than threshold, remove them altogether.

In [33]:
threshold = 0.001
vd_remove = wdAgg['vd'][wdAgg['vd']<threshold]
vs_remove = wdAgg['vs'][wdAgg['vs']<threshold]

Adjust demand in other industries up to cover this gap:

In [34]:
vd = adj.rc_pd(wdAgg['vd'], ('not', vd_remove))
vd_ = wdAgg['vd'].groupby(['m','k']).sum()
wdAgg['vd'] = vd * vd_/vd.groupby(['m','k']).sum()
vs = adj.rc_pd(wdAgg['vs'], ('not', vs_remove))
vs_ = wdAgg['vs'].groupby(['m','k']).sum()
wdAgg['vs'] = vs * vs_/vs.groupby(['m','k']).sum()

Adjust quantities similarly:

In [35]:
qd = adj.rc_pd(wdAgg['qd'] , ('not', vd_remove))
qd_ = wdAgg['qd'].groupby(['m','k']).sum()
wdAgg['qd'] = qd * qd_/qd.groupby(['m','k']).sum()
qs = adj.rc_pd(wdAgg['qs'] , ('not', vs_remove))
qs_ = wdAgg['qs'].groupby(['m','k']).sum()
wdAgg['qs'] = qs * qs_/qs.groupby(['m','k']).sum()

Check balances:

In [36]:
assert all([testBalance(wdAgg[f'{k}d'], wdAgg[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

## 2. Adjust IO with demand/supply of non IO goods

*Specify IO inputs to use in rebalancing. We include all production industries, foreign sector, and households. The waste sector produces energy from waste incineration and other services. We start by extracting the "waste incineration" part here and add it to the definition of the supply vector instead.*

### 2.1. Add distinction between waste/waste incineration

Initially, get the share of waste generation covered by data and the share of waste incineration in domestic plants:

In [37]:
sizeWTD = pd.DataFrame(wasteData['sizeWTD'].values)
sizeWTD = pd.DataFrame(sizeWTD.iloc[1:].values, columns = sizeWTD.iloc[0])
domWasteGenShare = adj.rc_pd(wdAgg['qs'].xs('W',level='k'), ('not',db('s_f'))).sum() / (sizeWTD['qWTD'][0]*10)
weShare = wd['qd'].xs(('383901','W'),level=('s','k')).sum() / (sizeWTD['qWTE'][0]*10)

Split up the demand for waste into energy from incineration and 'other' services:

In [38]:
vD_wasteE = weShare * db('vWE').copy()
vD_waste = adj.rc_pd(db('vD'), db('n_waste')).sub(vD_wasteE,fill_value = 0)
vD_wasteE.index = pyDatabases.cartesianProductIndex([vD_wasteE.index, db('n_wasteE')])
# # adjust supply vector as well:
vS_waste = vD_waste.groupby(['t','n']).sum()
vS_waste.index = stdSort(pyDatabases.cartesianProductIndex([vS_waste.index, db('sWaste')]))
vS_wasteE = vD_wasteE.groupby(['t','n']).sum()
vS_wasteE.index = stdSort(pyDatabases.cartesianProductIndex([vS_wasteE.index, db('sWaste')]))

Do the same for waste incineration from abroad (*note: No vS vector for F sector*):

In [39]:
vD_wasteF = adj.rc_pd(db('vD'), db('n_waste')+'_F').sub(db('vWE_F'), fill_value = 0)
vD_wasteEF = db('vWE_F').copy()
vD_wasteEF.index = pyDatabases.cartesianProductIndex([vD_wasteEF.index, db('n_wasteE')+'_F'])

Add demand vector without ```wasteE```:

In [40]:
vD0 = adj.rc_pd(db('vD').xs(t0), ('and', [db('n_p').union(db('n_F')), wdAgg['vd'].index.levels[0]]))
vD0 = vD_waste.xs(t0).combine_first(vD_wasteF.xs(t0)).combine_first(vD0)
vD0.index = vD0.index.remove_unused_levels()

### 2.2. Add materials demand

*Define demand for non-waste materials (values):*

In [41]:
vDn_df = wdAgg['vd'].rename('vD').reset_index()
vDn_df['n'] = vDn_df['m']+'_'+vDn_df['k']
vDnRaw = vDn_df.set_index(['s','n'])['vD']

*Supply:*

In [42]:
vSn_df = wdAgg['vs'].rename('vS').reset_index()
vSn_df['n'] = vSn_df['m']+'_'+vSn_df['k']
vSnRaw = vSn_df.set_index(['s','n'])['vS']

New types of traded goods to include:

In [43]:
nmTypes = pd.Index(vDn_df['m'].unique(), name = 'n') # material types
kmTypes = pd.Index(vDn_df['k'].unique(), name = 'k') # Recycled/virgin distinction
db['m'] = nmTypes.rename('m')
db['k'] = kmTypes
db['o'] = pd.Index(['D','F'], name = 'o')

Add combinations:

In [44]:
allCombinations = pd.MultiIndex.from_product([db('m'), db('k'), db('o')]).to_frame(index = False)
allCombinations['n'] = allCombinations['m']+'_'+allCombinations['k']
allCombinations.loc[allCombinations['o'] == 'F', 'n'] = allCombinations.loc[allCombinations['o'] == 'F', 'n']+'_F'
n2m2k2o_all = stdSort(pd.MultiIndex.from_frame(allCombinations))

### 2.3. Adjust IO data 

In [45]:
pd.set_option('future.no_silent_downcasting', True)
targets = CreateIOTargets(db)(vD0, vDnRaw, vSnRaw)
adjustedData = AdjustIO(ws = db.ws)(targets.db('vD0'), targets.db('vD'), targets.db('vDn'), 
                                    targets.db('vS0'), targets.db('vS'), targets.db('vSn'))

The ```AdjustIO``` class creates adjustments to IO data. For the model implementation later, we add a number of subsets to set up the nesting structure: 
* ```nm```: Full set of traded goods that requires demand/supply equations and equilibrium constraints.
* ```nm_F```: Foreign subset of ```nm```.
* ```nm_D```: Domestic subset of ```nm```.
* ```ns_F```: Foreign IO goods. *This includes waste incineration*.
* ```n_F```: All foreign goods (materials and IO goods).
* ```n2m2k2o```: Mapping.

In [46]:
db['nm'] = adjustedData('vSn').index.levels[-1] # materials
db['nm_F'] = adjustedData('vSn').xs('F').index # foreign materials
db['nm_D'] = db('nm').difference(db('nm_F')) # domestic materials
# adjust nm_F to cover all domestic materials as well
db['nm_F'] = db('nm_F').union(db('nm_D')+'_F')
db['ns_F'] = db('n_F').copy().union(vD_wasteEF.index.levels[-1]) # foreign IO materials
db['n_F'] = db('ns_F').union(db('nm_F')) # foreign goods
db['n2m2k2o'] = adj.rc_pd(n2m2k2o_all, db('nm'))

*Values:* 
* The demand vector is extended with demand for materials, energy from waste incineration, and existing "IO" demand is replaced with adjusted values. *Note: This vector does not include value of durables.*
* The supply vector is: (1) extended with materials and waste incineration, (2) existing IO supply is *adjusted* (we do not use ```vSsolve``` directly here, only changes, because it does not cover everything), (3) foreign supply is not used (it only enters the model through domestic demand for foreign goods). 

In [47]:
vDnew = pd.concat([stdSort(adjMultiIndex.bc(pd.concat([adjustedData('vDn'), adjustedData('vDsolve')], axis = 0), db('t0'))),
                   vD_wasteE, vD_wasteEF], axis = 0)
db['vD'] = vDnew.combine_first(db('vD'))
# supply:
vSadjustments = stdSort(adjMultiIndex.bc(adj.rc_pd(adjustedData('vS0')-adjustedData('vSsolve'), ('not', db('s_f'))), db('t0')))
vSadjustments = vSadjustments.add(vS_wasteE.droplevel('n'),fill_value=0)
vSio = adj.rc_pd(db('vS'), vSadjustments).sub(vSadjustments, fill_value=0)
vSn  = stdSort(adjMultiIndex.bc(adj.rc_pd(adjustedData('vSn'), ('not', db('s_f'))), db('t0')))
vS = pd.concat([vSio, vSn, vS_wasteE], axis = 0)
db['vS'] = vS.combine_first(db('vS'))

*Prices:* For prices, we use default prices (p = 1) for all IO goods including waste incineration. For materials, we use implied prices from materials data:

In [48]:
pm = (wdAgg['vs'] / wdAgg['qs']).dropna().rename('p').reset_index()
pm['n'] = pm['m']+'_'+pm['k']
pm.loc[pm['s'] == 'F','n'] = pm.loc[pm['s'] == 'F','n']+'_F'
pm = adj.rc_pd(pm.set_index('n')['p'], db('nm')).groupby('n').mean()
pm = stdSort(adjMultiIndex.bc(pm, db('t0')))
pns = stdSort(adjMultiIndex.bc(pd.Series(1, index = db('n_p').union(vD_wasteE.index.levels[-1]).union(db('ns_F'))), db('t0')))
pn = pd.concat([pm, pns], axis = 0)
pn = pd.concat([pn, adj.rc_pd(db('p'), ('not', pn))], axis = 0)
db['p'] = pn.combine_first(db('p')) # update prices

*Quantities:* Non-waste quantities are implied from values / prices. Waste quantities treated further below.

In [49]:
db['qD'] = (db('vD')/pn).dropna().combine_first(db('qD'))
db['qS'] = (db('vS')/pn).dropna().combine_first(db('qS'))

*Trade mappings:*

In [50]:
addGoods = db('nm_D').union(db('n_wasteE'))
db['dom2for'] = db('dom2for').union(pd.MultiIndex.from_arrays([addGoods, (addGoods+'_F').rename('nn')]))
db['dExport'] = adj.rc_pd(db('vD'), db('s_f')).index.droplevel('t').unique()
[db.series.__delitem__(k) for k in ('dImport', 'dImport_dom', 'dImport_for') if k in db.symbols];

*NB: If a service/material is not imported, the foreign good does not have any price in this vector. In this case, we input a price of 1 in the vector for the foreign good types (this makes the Armington work).*

In [51]:
db.aom(pd.Series(1, index = pd.MultiIndex.from_product([db('t0'), db('nm_F')])), name = 'p', priority='first')

## 3. Waste treatment data

For the waste treatment part, we start by collecting the data in a separate database. This is passed to solve the nonlinear program that estimates cost structure of the treatment industry. In the next section, we merge relevant data to the main database.

In [52]:
dbWaste = GpyDB(name = 'wasteDB', ws = db.ws)

### 3.1. Data inputs for cost estimation

#### A. Waste treatment splits

In [53]:
E = wd['qd'].xs(('383901','W'),level=('s','k')) # waste for incineration 
W = wdAgg['qd'].xs(('Waste','W'), level = ('s','k')) # total treated waste - this gives some problems with recycling efficiency > 100 %
A = wdAgg['qs'].xs(('Waste','R'),level=('s','k')) # recycled materials
dmin = pd.Series(0, index = db('m'))
D = dmin * W # landfill
R = W-E-D # waste going towards recycling
R̄ = W-D # Recyclable waste

Add to database:

In [54]:
dbWaste['E'] = gpy(E, type = 'par')
dbWaste['W'] = gpy(W, type = 'par')
dbWaste['A'] = gpy(A, type = 'par')
dbWaste['D'] = gpy(D, type = 'par')
dbWaste['dmin'] = gpy(dmin, type = 'par')
dbWaste['R'] = gpy(R, type = 'par')
dbWaste['Rbar'] = gpy(R̄, type = 'par')

#### B. Waste incineration parameter

Get relative incineration productivity from external data:

In [55]:
relative_we = pd.DataFrame(wasteData['relativeWE'].values)
relative_we = pd.DataFrame(relative_we.iloc[1:,].values, columns = relative_we.iloc[0]).set_index('m')['we']

Define scale to ensure price of one in baseline year:

In [56]:
γwe_scale = vS_wasteE.iloc[0] / (relative_we * E).sum() # assumes vS_wasteE only includes one data point
γwe = γwe_scale * relative_we # 

Add to database:*

In [57]:
# dbWaste['gamma_we'] = gpy(γwe, type = 'par')
dbWaste['gamma_we'] = γwe

Add total value of waste incineration (that our share of the waste captures):

In [58]:
dbWaste['valueWE'] = gpy((γwe * E).sum(), name = 'valueWE', type = 'par')

#### C. Domestically treated waste

Domestically treated waste is 100% of imports (of all $m$), and residual share of domestically produced waste to match total treatment:

In [59]:
wasteGen = wdAgg['qs'].xs('W',level='k')
wasteGen2WTD = adj.rc_pd(wasteGen, db('s_f'))

Share of domestic waste treated domestically:

In [60]:
domesticWasteGen = adj.rc_pd(wdAgg['qs'].xs('W',level='k'), ('not', db('s_f'))).groupby('m').sum()
domesticTreatmentDomWaste = wdAgg['qd'].xs(('Waste','W'),level=('s','k'))-wdAgg['qs'].xs(('F','W'),level=('s','k')) 
γwd = domesticTreatmentDomWaste/domesticWasteGen

Add this share to the wasteGen2WTD (waste generation going towards waste treatment domestic, WTD):

In [61]:
wasteGen2WTD = pd.concat([wasteGen2WTD, γwd * adj.rc_pd(wasteGen, ('not', db('s_f')))], axis = 0)
dbWaste['WGen2WTD'] = gpy(wasteGen2WTD, type= 'par')

#### D. Target waste treatment prices

In [62]:
pWTD = pd.DataFrame(wasteData['pWTD'].values)
pWTD = pd.DataFrame(pWTD.iloc[1:].values, columns = pWTD.iloc[0]).set_index('m')['pWTD']
pWTD = (pWTD / db._scale ) * 1e5 # Rescale prices to mia. DKK / 100.000 ton waste
dbWaste['pWTD'] = pWTD

#### E. Other

Compute expected expenditures related to waste management:

In [63]:
dbWaste['vDtarget'] = gpy((pWTD * dbWaste('WGen2WTD')).groupby('s').sum(), type = 'par')

Prices:

In [64]:
dbWaste['pe'] = gpy(db('p').xs((t0, 'WasteE')), name = 'pe', type = 'par')
dbWaste['pm'] = gpy((wdAgg['vs'] / wdAgg['qs']).dropna().xs('R',level='k').xs('Waste'), type = 'par')

Read sets:

In [65]:
AggDB.readSets(dbWaste, types = ['var','par'])

Smoothing parameter:

In [66]:
dbWaste['smooth'] = gpy(pd.Series(0.0001, index = dbWaste('m')), type = 'par')

Shape of recycling function:

In [67]:
dbWaste['beta'] = pd.Series(2, index = dbWaste('m'))

### 3.2. Estimate model

*Check out feasibility with different parameters:*

In [68]:
r = R/R̄
a = A/R
derivs = rates['dA/dR']
derivs.loc['Rubber'] = 0.2
# derivs.loc['Plastic'] = 0.03 # check

In [69]:
β = .35
al = ((1+β-r**β)/(β*(r**β))*(derivs-a*(1-r**β)/(1-r**β/(1+β)) ))
au = (derivs-al)/(1-r**β)+al

In [70]:
dbWaste['E'] = gpy(E, type = 'par')
dbWaste['W'] = gpy(W, type = 'par')
dbWaste['A'] = gpy(A, type = 'par')
dbWaste['D'] = gpy(D, type = 'par')
dbWaste['dmin'] = gpy(dmin, type = 'par')
dbWaste['R'] = gpy(R, type = 'par')
dbWaste['Rbar'] = gpy(R̄, type = 'par')
dbWaste['dAdR'] = gpy(derivs.rename('dAdR'), type = 'par')

Estimate full model from IO cost structure:

In [71]:
# mEst = EstimateTreatCosts(dbWaste)
# estDb = mEst(init = True);

Calibrate cost parameters to target waste management cost vector:

In [72]:
mEst = CalibToGR(dbWaste)
estDB = mEst(init = True)

### 3.3. Add as model data

#### A. Split up IO waste onto treatment services and residual

Firms and consumers demand waste residual services and waste treatment services. This is included in the index $n$ of traded goods (inputs in their respective industries). Here, we add the set of treatment services:

In [73]:
mw = estDB('W').index # waste materials
db['mw_D'] = mw # what materials are treated
db['nw_D'] = (mw+'_W').rename('n') # treatment services, domestically
db['nw_F'] = db('nw_D')+'_F' # treatment services, foreign firms
db['nw'] = db('nw_D').union(db('nw_F')) # all waste treatment services

Mapping from m2n:

In [74]:
n2m_dom = pd.MultiIndex.from_arrays([db('m')+'_W',db('m')], names = ['n','m'])
n2m_for = pd.MultiIndex.from_arrays([db('m')+'_W_F', db('m')], names = ['n','m'])
db['n2m_D'] = pd.MultiIndex.from_arrays([db('m')+'_W',db('m')], names = ['n','m'])
db['n2m_F'] = pd.MultiIndex.from_arrays([db('m')+'_W_F',db('m')], names = ['n','m'])
db['n2m'] = n2m_dom.union(n2m_for)
db['nr2m_D'] = pd.MultiIndex.from_arrays([db('m')+'_R', db('m')], names = ['n','m'])  # from recycled outputs to material type

Assume that prices on foreign treatment = prices on domestic treatment. Add this to the price vector:

In [75]:
p_nw = pd.concat([adjMultiIndex.applyMult(estDB('pWTD'), n2m_dom).droplevel('m'),
                  adjMultiIndex.applyMult(estDB('pWTD'), n2m_for).droplevel('m')], axis = 0)
db.aom(stdSort(adjMultiIndex.bc(p_nw, db('t0'))), name = 'p', priority = 'second')

Get total waste generation ($W_{t,i,m}$):

In [76]:
domesticGen = adj.rc_pd(estDB('WGen2WTD'), ('not', db('s_f'))) / γwd
foreignGen  = adj.rc_pd(estDB('WGen2WTD'), db('s_f'))
db['qWS'] = stdSort(adjMultiIndex.bc(pd.concat([domesticGen, foreignGen], axis = 0), db('t0')))
db['dWS'] = db('qWS').index.droplevel('t').unique()

```uWS_D[t,s,m]```: Share of waste $m$ from industry $s$ treated domestically.

In [77]:
db['uWS_D'] = pd.concat([adjMultiIndex.bc(γwd, adj.rc_pd(db('qWS'), ('not',db('s_f')))),
                         pd.Series(1, index = adj.rc_pd(db('qWS'), db('s_f')).index)], axis = 0)

How is waste treated - quantity (weight):

In [78]:
domTreat = adjMultiIndex.applyMult(db('qWS') * db('uWS_D'), db('n2m_D')).droplevel('m')
forTreat = adjMultiIndex.applyMult(adj.rc_pd(db('qWS') * (1-db('uWS_D')), ('not', db('s_f'))), db('n2m_F')).droplevel('m')
treatDem = stdSort(pd.concat([domTreat, forTreat], axis = 0))
db.aom(treatDem, name = 'qD', priority = 'second')
db['dWTn'] = treatDem.index.droplevel('t').unique()

Value of this demand for treatment services:

In [79]:
db.aom(stdSort(adj.rc_pd(db('qD'), db('nw')) * adj.rc_pd(db('p'), db('nw'))), name = 'vD', priority = 'second')

Adjust demands/supplies of residual services ```Waste``` and ```Waste_F``` net of treatment services. Adjust quantities as well:

In [80]:
vWasteResidual = adj.rc_pd(db('vD'), db('n_waste')).sub(adj.rc_pd(db('vD'), db('nw_D')).groupby(['t','s']).sum(), fill_value=0)
vWasteResidual_F = adj.rc_pd(db('vD'), db('n_waste')+'_F').sub(adj.rc_pd(db('vD'), db('nw_F')).groupby(['t','s']).sum(), fill_value=0)
db.aom(vWasteResidual.combine_first(vWasteResidual_F), name = 'vD', priority = 'second')
db.aom(adj.rc_pd(db('vS'), db('n_waste'))-dbWaste('vDtarget').sum(), name = 'vS', priority = 'second')

Update relevant quantities as well:

In [81]:
db.aom(adj.rc_pd(db('vD'), pd.Index(['Waste','Waste_F'], name = 'n')) / adj.rc_pd(db('p'), pd.Index(['Waste','Waste_F'],name='n')), 
       name = 'qD', priority = 'second')
db.aom(adj.rc_pd(db('vS'), pd.Index(['Waste'], name = 'n')) / adj.rc_pd(db('p'), pd.Index(['Waste'], name = 'n')),
       name = 'qS', priority='second')

#### B. Waste generation and waste management demand

**Total generation including internal recycling:**

*Recycled materials:*

In [82]:
qS_nm = adj.rc_pd(db('qS'), ('and', [db('nm_D'), ('not', db('sWaste'))])) # supply materials
qSr = adj.rc_pd(adjMultiIndex.applyMult(qS_nm, db('n2m2k2o')), pd.Index(['R'],name='k')).droplevel(['k','o']) # supply of recycled materials

*Assume internal recycling efficiency of waste management industry minus 10 p.p (or lowest feasible rate):*

In [83]:
bound = (qSr.droplevel('n')/(db('qWS').add(qSr.droplevel('n'), fill_value=0))).dropna() # boundary on recycling efficiency to have μ_int in [0,1]
internRCEff = adjMultiIndex.bc(rates['A/R']-0.1, bound).clip(bound,1)

*Compute implied share of waste going towards internal treatment:*

In [84]:
μW_int = (qSr.droplevel('n')/(internRCEff * db('qWS').add(qSr.droplevel('n'), fill_value=0))).dropna()

*Compute primary waste:*

In [85]:
qWSP = (db('qWS') /(1-μW_int * internRCEff)).dropna()

*Add symbols:*

In [86]:
db['intRcEff'] = internRCEff
db['uWS_int'] = μW_int
db['dWS_int'] = internRCEff.index.droplevel('t')
db['qWSP'] = (db('qWS') /(1-μW_int * internRCEff)).dropna()
db['dqSR'] = qSr.droplevel(['t','m']).index # recycling from internal process

**Waste generating intensities $\mu_{t,i,n,m}^W$:**
* Assume intensity of one for materials inputs = waste type. If inputs exceed waste --> set to ratio of inputs to waste generated
* For waste intensities where this does not cover all waste generation, assume a constant intensity across all other input types *in values* (except a few).

*Intensity of input types n = m:*

In [87]:
qDnm = adj.rc_pd(db('qD'), ('and', [db('nm'), ('not', db('s_f'))])) # demand for materials 
qDm = adjMultiIndex.applyMult(qDnm, db('n2m2k2o')).groupby(['t','s','m']).sum() # demand for material types (sum over source/origin)
wGen = db('qWSP').combine_first(adj.rc_pd(db('qWS'), ('not', db('s_f')))) # primary waste production
uWS_n2m_m = (wGen/qDm).dropna().clip(None, 1) # waste intensity for n == m
uWS_n2m_n = stdSort(adjMultiIndex.applyMult(uWS_n2m_m, db('n2m2k2o')).droplevel(['k','o']))
uWS_n2m_n = adj.rc_pd(uWS_n2m_n, db('qD')) # only keep if the demand is also there

*Waste generation intensities for other inputs:*

In [88]:
noWasteGen = pd.Index(['Energy','Energy_F','K','L','Waste','WasteE','Waste_F','WasteE_F'], name = 'n') # input types that does not generate waste
wGenRes = wGen.sub((uWS_n2m_m  * qDm).dropna(), fill_value=0) # waste generation, not yet accounted for
wGenRes = wGenRes[wGenRes>0]
qInp = adj.rc_pd(db('qD'), ('and', [('not', noWasteGen), ('not', db('s_f'))])) # all inputs that generates waste
qInp = adjMultiIndex.bc(adj.rc_pd(qInp, wGenRes), wGenRes)
qInp_xn2m = adj.rc_pd(qInp, ('not', db('n2m2k2o').droplevel(['o','k']))) # all inputs except n = m

*Compute values instead and define intensities:*

In [89]:
vInp_xn2m = (qInp_xn2m * db('p')).dropna()
vInt_m = wGenRes/vInp_xn2m.groupby(['t','s','m']).sum()
uvWS_xn2m = adjMultiIndex.bc(vInt_m, vInp_xn2m)
uWS_xn2m = (uvWS_xn2m * db('p')).dropna()
db['uWS'] = stdSort(pd.concat([uWS_n2m_n, uWS_xn2m], axis = 0))

What inputs generates waste in what industries, and what combinations of waste is it related to (dummies)?

In [90]:
db['dWSn'] = db('uWS').droplevel(['t','m']).index.unique()
db['dWSnm'] = db('uWS').droplevel('t').index.unique()

Residual waste management service demand: Note that a few $s$ does not generate waste (investment industry, inventory block), but has demand for waste services. For these industries, the demand for waste services is simply added to the conventional nesting structure:

In [91]:
db['dWTy'] = adj.rc_pd(db('qD'), ('and', [db('dWS').levels[0], pd.Index(['Waste'], name = 'n')])).index.droplevel('t').unique() # industries with demand for residual waste services tied to kton of waste generation
db['dWTyF'] = adj.rc_pd(db('qD'), ('and', [db('dWS').levels[0], pd.Index(['Waste_F'], name = 'n')])).index.droplevel('t').unique() # industries with demand for residual waste services tied to kton of waste generation
db['uWTy'] = stdSort(adj.rc_pd(db('qD'), db('dWTy'))/(db('uWS_D') * db('qWS')).groupby(['t','s']).sum()).xs(t0) 
db['uWTyF'] = stdSort(adj.rc_pd(db('qD'), db('dWTyF'))/adj.rc_pd((1-db('uWS_D'))*db('qWS'), db('dWTyF')).groupby(['t','s']).sum()).xs(t0)

#### C. Domestic waste treatment firm

*This goes through section in the model documentation and creates relevant variables/parameters.*

An overview of model variables/parameter/dummies and counterparts in documentation:
* ```WTD_W[t,m]``` $= W_{t,wtd,m}^{in}$: Waste of type $m$ treated by domestic firm.
* ```WTD_d[t,m]``` $= D_{t,m}/W_{t,wtd,m}^{in}$: Share of waste going to landfill (domestic treatment firm).
* ```WTD_dmin[t,m]```  $=\underline{d}_{t,m}$: Minimum share of waste going to landfill.
* ```WTD_r[t,m]``` $= R_{t,m}/\overline{R}_{t,m}$: Share of recyclable waste going to recycling.
* ```WTD_a[t,m]``` $= A_{t,m}/R_{t,m}$: Recycling efficiency for recycled waste.
* ```WTD_e[t,m]``` $= E_{t,m}/W_{t,wtd,m}^{in}$: Share of waste going to waste incineration.
* ```WTD_gamwe[m]``` $=\gamma_{t,m}^{we}$: Productivity of waste for incineration (converts from kton waste to PJ)
* ```WTD_alphal[t,m]```$=\underline{\alpha}_{t,m}$: Lower bound on recycling quality.
* ```WTD_alphau[t,m]```$=\overline{\alpha}_{t,m}$:  Upper bound on recycling quality.
* ```WTD_beta[t,m]```  $=\beta_{t,m}$: Shape of recycling function.
* ```WTD_smooth[m]```: Smoothing parameter on interior function.
* ```WTD_ge[t,m], WTD_gd[t,m], WTD_gr[t,m]```: $\gamma_{t,m}^x$ cost parameters in recycling function.
* ```WTD_zetaE[t,m]```: Aux. variable.

Model variables like $\zeta_{t,m}, p_{t,zw}$ etc. are initialized in the waste module. 

In [92]:
db['WTD_W'] = (db('qWS') * db('uWS_D')).groupby(['t','m']).sum()
db['WTD_d'] = stdSort(adjMultiIndex.bc(D/estDB('W'), db('t0')))
db['WTD_dmin']  = stdSort(adjMultiIndex.bc(dmin, db('t0')))
db['WTD_r'] = stdSort(adjMultiIndex.bc(estDB('R')/(estDB('W')*(1-dmin)) , db('t0')))
db['WTD_a'] = stdSort(adjMultiIndex.bc(estDB('A')/estDB('R') , db('t0')))
db['WTD_e'] = stdSort(adjMultiIndex.bc(estDB('E')/estDB('W'), db('t0')))
db['WTD_gwe'] = estDB('gamma_we')
db['WTD_alphal'] = stdSort(adjMultiIndex.bc(estDB('alpha_l'), db('t0')))
db['WTD_alphau'] = stdSort(adjMultiIndex.bc(estDB('alpha_u'), db('t0')))
db['WTD_beta']   = stdSort(adjMultiIndex.bc(estDB('beta'), db('t0')))
db['WTD_smooth'] = estDB('smooth')
db['WTD_gd'] = stdSort(adjMultiIndex.bc(estDB('gamma_d'), db('t0')))
db['WTD_ge'] = stdSort(adjMultiIndex.bc(estDB('gamma_e'), db('t0')))
db['WTD_gr'] = stdSort(adjMultiIndex.bc(estDB('gamma_r'), db('t0')))
db['WTD_zetaE'] = stdSort(adjMultiIndex.bc(estDB('zetaE'), db('t0')))

Add supply of treatment services to the database as well:

In [93]:
qS_nw = adjMultiIndex.applyMult(db('WTD_W'), db('n2m_D')).droplevel('m')
db.aom(stdSort(adjMultiIndex.bc(qS_nw, wsIdx)), name = 'qS', priority = 'second') # qS
db.aom(stdSort(adjMultiIndex.bc(qS_nw * adj.rc_pd(db('p'), db('nw_D')), wsIdx)), name = 'vS', priority = 'second') # vS

#### D. Dummies/subsets

*Dummies:*

In [94]:
[db.__setitem__(k, db(k)[db(k) != 0]) for k in ('vS','qS','vD','qD')]; # remove zero values
db['nEqui'] = adj.rc_pd(db('vS').xs(t0).index.droplevel('s').unique(), ('not', db('nw_D'))) # what goods require an equilibrium condition
db['d_qS'] = adj.rc_pd(db('vS').xs(t0).index, ('not', db('nw_D'))) 
db['d_qD'] = adj.rc_pd(db('vD').xs(t0), db('nEqui')).index
db['d_pEqui'] = pd.Index(['L'], name ='n').union(db('nm_D').union(db('n_wasteE'))) # Subset of prices to be endogenized in general equilibrium. 
db['endo_qS'] = adj.rc_pd(db('d_qS'), db('nm_D')).unique() # quantities that are endogenous in baseline. 

*Trade mappings:*

In [95]:
db['dom2for'] = db('dom2for').union(pd.MultiIndex.from_arrays([db('nw_D'), db('nw_F').rename('nn')]))
db['dExport'] = adj.rc_pd(db('vD'), db('s_f')).index.droplevel('t').unique()

## 4. Production module

*Does not include waste industry.*

In [96]:
sIdx = db('s_p').difference(wsIdx)

### 4.1. Nesting structure

In general, the nesting structure *without* the materials/generic service demand is specified as:

In [97]:
mu = pd.MultiIndex.from_tuples([('KELM','RxE'),('KELM','KEL'),
                                ('KEL','L'), ('KEL','KE'),
                                ('KE','K'), ('KE','E'),
                                ('E','WE'), ('E', 'ExWE'), 
                                ('WE','WasteE'), ('WE','WasteE_F'),
                                ('ExWE','Energy'),('ExWE','Energy_F'),
                                ('RxE', 'ZY'), ('RxE', 'ZO')], names = ['n','nn'])

Here, ```ZY``` is consists of IO-specific *services*. This *does not* include energy and waste services: Energy is included in the other "leg" of the nesting tree; Waste services are modelled specifically in the waste generation module below. Each of these are then (potentially) split into foreign and domestic components:

In [98]:
ns = db('n_p').difference(pd.Index(['Energy','Waste'], name = 'n'))
mZY = pd.MultiIndex.from_product([['ZY'], 'RxE_'+ns], names = ['n','nn'])
mZYD = pd.MultiIndex.from_arrays([mZY.levels[-1], ns], names = ['n','nn'])
mZYF = pd.MultiIndex.from_arrays([mZY.levels[-1], ns+'_F'], names = ['n','nn'])

The nest with other materials is split into aggregates for each materials' type. Then, each type is split into virgin/recycled. Finally, each of these are split into domestic/foreign origin:

In [99]:
mZO = pd.MultiIndex.from_product([['ZO'], 'RxE_'+nmTypes], names = ['n','nn'])
mZOV = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_V'], names = ['n','nn'])
mZOW = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_R'], names = ['n','nn'])
nmV = mZOV.levels[-1].str.slice(4) # remove RxE_
nmW = mZOW.levels[-1].str.slice(4) # remove RxE_
mZOVD = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV], names = ['n','nn'])
mZOVF = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV+'_F'], names = ['n','nn'])
mZOWD = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW], names = ['n','nn'])
mZOWF = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW+'_F'], names = ['n','nn'])

Collect in one map and add full sector index to begin with:

In [100]:
mFull = reduce(pd.Index.union, [mu, mZY, mZYD, mZYF, mZO, mZOV, mZOW, mZOVD, mZOVF, mZOWD, mZOWF])
m = pyDatabases.cartesianProductIndex([sIdx, mFull])

Next, we "trim" the nesting tree - i.e. we remove redundant nests. *NB: This only works with pure "input trees".*

In [101]:
sparsity = adj.rc_pd(db('vD')[db('vD')!=0], sIdx).droplevel('t').index
m = nestingTree.trimNestingStructure(m, sparsity)

Now, identify the domestic goods that could potential also be part of the output structure and add a temporary '_input' to the names:

In [102]:
n = db('nm_D').union(db('n_p')).rename('nn')
mInp = pd.MultiIndex.from_arrays([n, (n+'_input').rename('ntemp')])
m = adj.rc_pd(m, ('not', n)).union(adjMultiIndex.applyMult(m, mInp).droplevel('nn').rename(['s','n','nn']))

* For the output part, add a simple nest to the sector specific output + vector of materials outputs.
* Note that the nesting trees are always specified with (n,nn) indicating $n$ linking to a lower node $nn$. Thus, for output trees, (n,nn) maps from branches (n) knots (nn), while the reverse is true for input trees.
* Only add virgin material production to this nesting structure; recycled materials are added separately in this formulation.

From KELM to sector outputs and materials:

In [103]:
mns = pyDatabases.cartesianProductIndex([pd.MultiIndex.from_arrays([sIdx, sIdx.rename('n')]), pd.Index(['KELM'], name = 'nn')])
mnm = pyDatabases.cartesianProductIndex([adj.rc_pd(db('qS'), ('and', [db('nm_D'), sIdx, ('not', db('dqSR'))])).xs(t0).index, pd.Index(['KELM'], name = 'nn')])

Save nesting trees:

In [104]:
db['nestProdInp'] = m
db['nestProdOut'] = mns.union(mnm)

### 4.2. Elasticities

Here, we assign generic elasticities depending on the placement in the nesting. Generally, we let elasticity of substitution be fairly high between domestic/foreign and recycled/virgin materials. Across different types of materials, we use a fairly low elasticity.

*CET elasticity (absolute value)*

In [105]:
etaOut = pd.Series(0.1, index = db('nestProdOut').droplevel('n').unique().rename(['s','n']))

*CES elasticities, non-materials:*

In [106]:
sigmaKELM = pd.Series(0.64, index  = m[m.get_level_values('n')=='KELM'].droplevel('nn').unique(), name = 'sigma')
sigmaKEL = pd.Series(0.73, index  = m[m.get_level_values('n')=='KEL'].droplevel('nn').unique(), name = 'sigma')
sigmaKE = pd.Series(0.1, index = m[m.get_level_values('n') == 'KE'].droplevel('nn').unique(), name = 'sigma')
sigmaE = pd.Series(2, index  = m[m.get_level_values('n')=='E'].droplevel('nn').unique(), name = 'sigma')
sigmaWE = pd.Series(3, index  = m[m.get_level_values('n')=='WE'].droplevel('nn').unique(), name = 'sigma')
sigmaExWE = pd.Series(3, index  = m[m.get_level_values('n')=='ExWE'].droplevel('nn').unique(), name = 'sigma')

*CES materials:*

In [107]:
sigmaRxE = pd.Series(0.64, index = m[m.get_level_values('n')=='RxE'].droplevel('nn').unique(), name = 'sigma')
sigmaZY = pd.Series(0.05, index = m[m.get_level_values('n')=='ZY'].droplevel('nn').unique(), name = 'sigma')
sigmaZYym = pd.Series(3, index = m[m.get_level_values('n').isin('RxE_'+ns)].droplevel('nn').unique(), name = 'sigma')
sigmaZO = pd.Series(0.1, index = m[m.get_level_values('n')=='ZO'].droplevel('nn').unique(), name = 'sigma')
sigmaZOnm = pd.Series(3, index = m[m.get_level_values('n').isin('RxE_'+nmTypes)].droplevel('nn').unique(), name = 'sigma')
sigmaZOVW = pd.Series(3, index = m[m.get_level_values('n').isin(('RxE_'+nmTypes+'_V').union('RxE_'+nmTypes+'_R'))].droplevel('nn').unique(), name = 'sigma')

Add to database:

In [108]:
sigma = pd.concat([sigmaKELM, sigmaKEL, sigmaKE, sigmaE, sigmaWE, sigmaExWE,sigmaRxE, sigmaZY, sigmaZYym, sigmaZO, sigmaZOnm, sigmaZOVW], axis = 0 )
db.aom(sigma, name = 'sigma') # elasticity of substitution
db.aom(etaOut, name = 'eta') # elasticity of transformation

### 4.3. Regulation and prices

Define output and input shares in values. Map CO2 emissions to output shares (do this for the waste treatment industry as well):

In [109]:
output = adj.rc_pd(db('vS'), db('s_p'))
inputs = adj.rc_pd(db('vD'), ('and', [db('s_p'), ('not', db('dur_p'))]))
outShares = output/pyDatabases.pdSum(output, 'n') # output shares
db['qCO2'] = db('qCO2') * outShares # overwrite qCO2 to be split into potentially multiple outputs from each sector

Define tax rate on emissions energy and environmental taxes. Broadcast sector-average rate onto all outputs with CO2 emissions:

In [110]:
db['tauCO2'] = adjMultiIndex.bc(db('vTax').xs('Emissions',level='taxTypes') / pyDatabases.pdSum(db('qCO2'), 'n'), db('qCO2'))
db['dtauCO2'] = db('tauCO2').index.droplevel('t').unique()
db['dqCO2'] = db('qCO2').index.droplevel('t').unique()

Add the rest of the taxes to a sector-specific lump sum tax and set VAT inputs/output taxes to zero:

In [111]:
db.aom(adj.rc_pd(db('TotalTax'), db['s_p'])-db('vTax').xs('Emissions',level='taxTypes'), name = 'tauLump')
db.aom(pd.Series(0, index = output.index), name = 'tauS')
db.aom(pd.Series(0, index = inputs.index), name = 'tauD')

Define marginal waste management cost index for inputs:

In [112]:
# Mc of domestic waste treatment per waste m:
mcWTD = (db('uWS_D') * adjMultiIndex.applyMult(adj.rc_pd(db('p'), db('nw_D')), db('n2m_D'))).groupby(['t','s','m']).sum()
# Mc of foreign waste treatment per marginal increase in input type n:
mcWTF = ((1-db('uWS_D')) * adjMultiIndex.applyMult(adj.rc_pd(db('p'), db('nw_F')), db('n2m_F'))).groupby(['t','s','m']).sum()
# Mc of residual waste services per marginal increase waste m:
mcWTy = db('uWS_D') * db('uWTy').droplevel('n')
# Mc of foreign residual waste services per marginal increase in m:
mcWTyF = ((1-db('uWS_D')) * db('uWTyF').droplevel('n')).dropna()
# External treatment price:
pWext = mcWTD.add(mcWTF, fill_value=0).add(mcWTy, fill_value=0).add(mcWTyF, fill_value=0)

Define marginal costs per input $n$:

In [113]:
uWSext = db('uWS').mul((1-db('uWS_int')*db('intRcEff')),fill_value=1)
prm = adjMultiIndex.applyMult(adj.rc_pd(db('p'), db('nr2m_D')), db('nr2m_D')).droplevel('n') # price of recycled materials 
mrRC = ((db('uWS') * db('uWS_int')*db('intRcEff')).dropna() * prm).dropna().groupby(['t','s','n']).sum()
pW = (pWext * uWSext).groupby(['t','s','n']).sum().sub(mrRC, fill_value=0)

Define sector-specific prices given regulation:

In [114]:
db.aom(adj.rc_pd(pWext, sIdx), name = 'pWext',priority = 'second')
db.aom(pW, name = 'pW', priority = 'second')
db.aom(stdSort(adj.rc_pd((1+db('tauD'))*db('p'), inputs)).add(adj.rc_pd(pW, db('s_p')), fill_value=0), name = 'pD')

## 5. Waste Management Module

*Does not include waste industry.*

The waste treatment module is somewhat different. The first part, however, is a somewhat standard nested CES/CET production structure. However, how the waste treatment aggregate $Z_t^W$ is disaggregated further is modelled in an atypical way. These final parts are added in the model class. 

### 5.1. Nesting structure

In general, the nesting structure *without* the materials/generic service demand is specified as:

In [115]:
mu = pd.MultiIndex.from_tuples([('KELM','RxE'),('KELM','KEL'),
                                ('KEL','L'), ('KEL','KE'),
                                ('KE','K'), ('KE','E'),
                                ('E','WE'), ('E', 'ExWE'), 
                                ('WE','WasteE'), ('WE','WasteE_F'),
                                ('ExWE','Energy'),('ExWE','Energy_F'),
                                ('RxE', 'ZY'), ('RxE', 'ZO')], names = ['n','nn'])

Here, ```ZY``` is consists of IO-specific *services*. This *does not* include energy and waste services: Energy is included in the other "leg" of the nesting tree; Waste services are modelled specifically in the waste generation module below. Each of these are then (potentially) split into foreign and domestic components:

In [116]:
ns = db('n_p').difference(pd.Index(['Energy','Waste'], name = 'n'))
mZY = pd.MultiIndex.from_product([['ZY'], 'RxE_'+ns], names = ['n','nn'])
mZYD = pd.MultiIndex.from_arrays([mZY.levels[-1], ns], names = ['n','nn'])
mZYF = pd.MultiIndex.from_arrays([mZY.levels[-1], ns+'_F'], names = ['n','nn'])

The nest with other materials is split into aggregates for each materials' type. Then, each type is split into virgin/recycled. Finally, each of these are split into domestic/foreign origin:

In [117]:
mZO = pd.MultiIndex.from_product([['ZO'], 'RxE_'+nmTypes], names = ['n','nn'])
mZOV = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_V'], names = ['n','nn'])
mZOW = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_R'], names = ['n','nn'])
nmV = mZOV.levels[-1].str.slice(4) # remove RxE_
nmW = mZOW.levels[-1].str.slice(4) # remove RxE_
mZOVD = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV], names = ['n','nn'])
mZOVF = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV+'_F'], names = ['n','nn'])
mZOWD = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW], names = ['n','nn'])
mZOWF = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW+'_F'], names = ['n','nn'])

Collect in one map and add full sector index to begin with:

In [118]:
mFull = reduce(pd.Index.union, [mu, mZY, mZYD, mZYF, mZO, mZOV, mZOW, mZOVD, mZOVF, mZOWD, mZOWF])
m = pyDatabases.cartesianProductIndex([wsIdx, mFull])

Next, we "trim" the nesting tree - i.e. we remove redundant nests. *NB: This only works with pure "input trees".*

In [119]:
sparsity = adj.rc_pd(db('vD')[db('vD')!=0], wsIdx).droplevel('t').index
m = nestingTree.trimNestingStructure(m, sparsity)

Now, identify the domestic goods that could potential also be part of the output structure and add a temporary '_input' to the names:

In [120]:
n = db('nm_D').union(db('n_p')).rename('nn')
mInp = pd.MultiIndex.from_arrays([n, (n+'_input').rename('ntemp')])
m = adj.rc_pd(m, ('not', n)).union(adjMultiIndex.applyMult(m, mInp).droplevel('nn').rename(['s','n','nn']))

For the output part, here we end with splitting into the residual service and a waste aggregate that we'll simply reference ```ZW```:

In [121]:
mns = pyDatabases.cartesianProductIndex([wsIdx, pd.Index(['Waste','ZW'], name = 'n'), pd.Index(['KELM'], name = 'nn')])

*NB: We add an additional branch on top of the ZW knot that is removed in the process of initializing the model.*

In [122]:
# mns = mns.union(pd.MultiIndex.from_arrays([wsIdx, ['DELBRANCH'], ['ZW']], names = ['s','n','nn']))

Save nesting trees:

In [123]:
db['nestWasteInp'] = m
db['nestWasteOut'] = mns

*Special subsets:*

In [124]:
db['n_wasteE'] = pd.Index(['WasteE'], name = 'n') # waste incineration output
db['n_ZW'] = pd.Index(['ZW'], name = 'n') # note tied to waste treatment production technology

### 5.2. Elasticities

*CET elasticity (absolute value)*

In [125]:
etaOut = pd.Series(0.25, index = db('nestWasteOut').droplevel('n').unique().rename(['s','n']))

*CES elasticities, non-materials:*

In [126]:
sigmaKELM = pd.Series(0.64, index  = m[m.get_level_values('n')=='KELM'].droplevel('nn').unique(), name = 'sigma')
sigmaKEL = pd.Series(0.73, index  = m[m.get_level_values('n')=='KEL'].droplevel('nn').unique(), name = 'sigma')
sigmaKE = pd.Series(0.1, index = m[m.get_level_values('n') == 'KE'].droplevel('nn').unique(), name = 'sigma')
sigmaE = pd.Series(2, index  = m[m.get_level_values('n')=='E'].droplevel('nn').unique(), name = 'sigma')
sigmaWE = pd.Series(3, index  = m[m.get_level_values('n')=='WE'].droplevel('nn').unique(), name = 'sigma')
sigmaExWE = pd.Series(3, index  = m[m.get_level_values('n')=='ExWE'].droplevel('nn').unique(), name = 'sigma')

*CES materials:*

In [127]:
sigmaRxE = pd.Series(0.64, index = m[m.get_level_values('n')=='RxE'].droplevel('nn').unique(), name = 'sigma')
sigmaZY = pd.Series(0.05, index = m[m.get_level_values('n')=='ZY'].droplevel('nn').unique(), name = 'sigma')
sigmaZYym = pd.Series(3, index = m[m.get_level_values('n').isin('RxE_'+ns)].droplevel('nn').unique(), name = 'sigma')
sigmaZO = pd.Series(0.1, index = m[m.get_level_values('n')=='ZO'].droplevel('nn').unique(), name = 'sigma')
sigmaZOnm = pd.Series(3, index = m[m.get_level_values('n').isin('RxE_'+nmTypes)].droplevel('nn').unique(), name = 'sigma')
sigmaZOVW = pd.Series(3, index = m[m.get_level_values('n').isin(('RxE_'+nmTypes+'_V').union('RxE_'+nmTypes+'_R'))].droplevel('nn').unique(), name = 'sigma')

Add to database:

In [128]:
sigma = pd.concat([sigmaKELM, sigmaKEL, sigmaKE, sigmaE, sigmaWE, sigmaExWE,sigmaRxE, sigmaZY, sigmaZYym, sigmaZO, sigmaZOnm, sigmaZOVW], axis = 0 )
db.aom(sigma, name = 'sigma') # elasticity of substitution
db.aom(etaOut, name = 'eta') # elasticity of transformation

## 6. Investment module

In [129]:
sIdx = db('s_i')

### 6.1. Nesting structure

The investment firm relies on materials and service inputs, but no durables nor labor. Similarly, they do not generate waste for treatment. The nesting tree is somewhat more simple here: As with the production firm, they have a materials and a service nest that is combined into the relevant output (investment good).

In [130]:
mu = pd.MultiIndex.from_tuples([('Y','ZY'), ('Y','ZO'), ('Y','ZE')], names = ['n','nn'])

*Service demands:*

In [131]:
ns = db('n_p').difference(pd.Index(['Energy'],name='n')) # goods included in the "service" branch
mZY = pd.MultiIndex.from_product([['ZY'], 'ZY_'+ns], names = ['n','nn']) # service aggregates
mZYD = pd.MultiIndex.from_arrays([mZY.levels[-1], ns], names = ['n','nn']) # domestic service demand
mZYF = pd.MultiIndex.from_arrays([mZY.levels[-1], ns+'_F'], names = ['n','nn']) # foreign service demand

*Materials nest: Split sequentially into material types, virgin/recycled, domestic/foreign orign:*

In [132]:
mZO = pd.MultiIndex.from_product([['ZO'], 'RxE_'+nmTypes], names = ['n','nn'])
mZOV = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_V'], names = ['n','nn'])
mZOW = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_R'], names = ['n','nn'])
nmV = mZOV.levels[-1].str.slice(4) # remove RxE_
nmW = mZOW.levels[-1].str.slice(4) # remove RxE_
mZOVD = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV], names = ['n','nn'])
mZOVF = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV+'_F'], names = ['n','nn'])
mZOWD = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW], names = ['n','nn'])
mZOWF = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW+'_F'], names = ['n','nn'])

*Energy nest:*

In [133]:
ne = pd.Index(['Energy','WasteE'], name = 'n')
mZE = pd.MultiIndex.from_product([['ZE'], 'ZE_'+ne], names = ['n','nn'])
mZED = pd.MultiIndex.from_arrays([mZE.levels[-1], mZE.levels[-1].str.slice(3)], names = ['n','nn'])
mZEF = pd.MultiIndex.from_arrays([mZE.levels[-1], mZE.levels[-1].str.slice(3)+'_F'], names = ['n','nn'])

Collect in one mapping and add sector index:

In [134]:
mFull = reduce(pd.Index.union, [mu, mZY, mZYD, mZYF, mZO, mZOV, mZOW, mZOVD, mZOVF, mZOWD, mZOWF, mZE, mZED, mZEF])
m = pyDatabases.cartesianProductIndex([sIdx, mFull])

Next, we "trim" the nesting tree - i.e. we remove redundant nests. *NB: This only works with pure "input trees".*

In [135]:
sparsity = adj.rc_pd(db('vD')[db('vD')!=0], sIdx).droplevel('t').index
m = nestingTree.trimNestingStructure(m, sparsity)

Replace "Y" with sector output:

In [136]:
df = m.to_frame(index=False)
df.loc[df.n == 'Y','n'] = df.loc[df.n == 'Y','s']
m = pd.MultiIndex.from_frame(df)

Save nesting tree:

In [137]:
db['nestInvest'] = m

### 6.2. Elasticities

*CES upper nest:*

In [138]:
sigmaY = pd.Series(0.1, index = m[m.get_level_values('n').isin(sIdx.rename('n'))].droplevel('nn').unique(), name = 'sigma')

*CES, materials:*

In [139]:
sigmaZO = pd.Series(0.1, index = m[m.get_level_values('n')=='ZO'].droplevel('nn').unique(), name = 'sigma')
sigmaZOnm = pd.Series(3, index = m[m.get_level_values('n').isin('RxE_'+nmTypes)].droplevel('nn').unique(), name = 'sigma')
sigmaZOVW = pd.Series(3, index = m[m.get_level_values('n').isin(('RxE_'+nmTypes+'_V').union('RxE_'+nmTypes+'_R'))].droplevel('nn').unique(), name = 'sigma')

*CES elasticities, services:*

In [140]:
sigmaZY = pd.Series(0.1, index = m[m.get_level_values('n') == 'ZY'].droplevel('nn').unique(), name = 'sigma')
sigmaZYym = pd.Series(3, index = m[m.get_level_values('n').isin('ZY_'+ns)].droplevel('nn').unique(), name = 'sigma')

*CES elasticities, energy:*

In [141]:
sigmaZE = pd.Series(0.1, index = m[m.get_level_values('n') == 'ZE'].droplevel('nn').unique(), name = 'sigma')
sigmaZEym = pd.Series(3, index = m[m.get_level_values('n').isin('ZE_'+ne)].droplevel('nn').unique(), name = 'sigma')

Add to database:

In [142]:
sigma = pd.concat([sigmaY, sigmaZO, sigmaZOnm, sigmaZOVW, sigmaZY, sigmaZYym, sigmaZE, sigmaZEym], axis =0)
db.aom(sigma, name = 'sigma') # elasticity of substitution

### 6.3. Regulation and prices

The investment sector does not itself emit emissions; however, investments are still tied to emissions through this sectors' reliance on inputs from other sectors (that are emission intensive).

In [143]:
output = adj.rc_pd(db['vS'], db['s_i']) # output
inputs = adj.rc_pd(db['vD'], db['s_i']) # inputs
outShares = output/pyDatabases.pdSum(output, 'n') # output shares

The rest of the regulation is straightforward; we assume that all taxes are paid on output:

In [144]:
db.aom(outShares * adj.rc_pd(db('TotalTax'), db['s_i'])/output, name = 'tauS')
db.aom(adj.rc_pd(db('TotalTax'), db['s_i'])*0, name = 'tauLump')
db.aom(pd.Series(0, index = inputs.index), name = 'tauD')

Define sector-specific prices given regulation (this does not include taxes on emissions or waste generation):

In [145]:
db.aom(stdSort(adj.rc_pd((1+db('tauD'))*db('p'), inputs)), name = 'pD')

## 7. Households

Households demand materials and services like the investment module. Beyond this, they demand waste treatment services as they too generate waste. Beyond this, they supply labor and save/invest over time.

In [146]:
sIdx = db('s_HH')

### 7.1. Nesting structure

Cosnumption aggregate to services, materials, and energy (very similar to investment firm):

In [147]:
mu = pd.MultiIndex.from_tuples([('C','ZY'), ('C','ZO'), ('C','ZE')], names = ['n','nn'])

*Service demands:*

In [148]:
ns = db('n_p').difference(pd.Index(['Energy','Waste'],name='n')) # goods included in the "service" branch
mZY = pd.MultiIndex.from_product([['ZY'], 'ZY_'+ns], names = ['n','nn']) # service aggregates
mZYD = pd.MultiIndex.from_arrays([mZY.levels[-1], ns], names = ['n','nn']) # domestic service demand
mZYF = pd.MultiIndex.from_arrays([mZY.levels[-1], ns+'_F'], names = ['n','nn']) # foreign service demand

*Materials nest: Split sequentially into material types, virgin/recycled, domestic/foreign orign:*

In [149]:
mZO = pd.MultiIndex.from_product([['ZO'], 'RxE_'+nmTypes], names = ['n','nn'])
mZOV = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_V'], names = ['n','nn'])
mZOW = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_R'], names = ['n','nn'])
nmV = mZOV.levels[-1].str.slice(4) # remove RxE_
nmW = mZOW.levels[-1].str.slice(4) # remove RxE_
mZOVD = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV], names = ['n','nn'])
mZOVF = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV+'_F'], names = ['n','nn'])
mZOWD = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW], names = ['n','nn'])
mZOWF = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW+'_F'], names = ['n','nn'])

*Energy nest:*

In [150]:
ne = pd.Index(['Energy','WasteE'], name = 'n')
mZE = pd.MultiIndex.from_product([['ZE'], 'ZE_'+ne], names = ['n','nn'])
mZED = pd.MultiIndex.from_arrays([mZE.levels[-1], mZE.levels[-1].str.slice(3)], names = ['n','nn'])
mZEF = pd.MultiIndex.from_arrays([mZE.levels[-1], mZE.levels[-1].str.slice(3)+'_F'], names = ['n','nn'])

Collect in one mapping and add sector index:

In [151]:
mFull = reduce(pd.Index.union, [mu, mZY, mZYD, mZYF, mZO, mZOV, mZOW, mZOVD, mZOVF, mZOWD, mZOWF, mZE, mZED, mZEF])
m = pyDatabases.cartesianProductIndex([sIdx, mFull])

Next, we "trim" the nesting tree - i.e. we remove redundant nests. *NB: This only works with pure "input trees".*

In [152]:
sparsity = adj.rc_pd(db('vD')[db('vD')!=0], sIdx).droplevel('t').index
m = nestingTree.trimNestingStructure(m, sparsity)

Save nesting tree:

In [153]:
db['nestHH'] = m

Add mapping from consumption aggregate to labor:

In [154]:
db['L2C'] = pd.MultiIndex.from_arrays([db('s_HH'), pd.Index(['L']*len(db('s_HH')), name = 'n'), pd.Index(['C']*len(db('s_HH')), name = 'nn')])

### 7.2. Elasticities:

*CES upper nest:*

In [155]:
sigmaC = pd.Series(0.5, index = m[m.get_level_values('n') == 'C'].droplevel('nn').unique(), name = 'sigma')

*CES, materials:*

In [156]:
sigmaZO = pd.Series(0.25, index = m[m.get_level_values('n')=='ZO'].droplevel('nn').unique(), name = 'sigma')
sigmaZOnm = pd.Series(3, index = m[m.get_level_values('n').isin('RxE_'+nmTypes)].droplevel('nn').unique(), name = 'sigma')
sigmaZOVW = pd.Series(3, index = m[m.get_level_values('n').isin(('RxE_'+nmTypes+'_V').union('RxE_'+nmTypes+'_R'))].droplevel('nn').unique(), name = 'sigma')

*CES elasticities, services:*

In [157]:
sigmaZY = pd.Series(0.25, index = m[m.get_level_values('n') == 'ZY'].droplevel('nn').unique(), name = 'sigma')
sigmaZYym = pd.Series(3, index = m[m.get_level_values('n').isin('ZY_'+ns)].droplevel('nn').unique(), name = 'sigma')

*CES elasticities, energy:*

In [158]:
sigmaZE = pd.Series(0.25, index = m[m.get_level_values('n') == 'ZE'].droplevel('nn').unique(), name = 'sigma')
sigmaZEym = pd.Series(3, index = m[m.get_level_values('n').isin('ZE_'+ne)].droplevel('nn').unique(), name = 'sigma')

Add to database:

In [159]:
sigma = pd.concat([sigmaC, sigmaZO, sigmaZOnm, sigmaZOVW, sigmaZY, sigmaZYym, sigmaZE, sigmaZEym], axis =0)
db.aom(sigma, name = 'sigma') # elasticity of substitution

Frisch elasticity, CRRA, time preferences:

In [160]:
db.aom(pd.Series(0.1, index = db('s_HH')), name = 'frisch')
db.aom(pd.Series(2, index = db('s_HH')), name = 'crra')
db.aom(pd.Series((1+db('g_LR'))**(db('crra'))/db('R_LR'), index = db('s_HH')), name = 'discF')

### 7.3. Regulation:

Households do not directly emit anything (it all happens in the production process). Currently, we use a very simple type of regulation here:
* Households pay VAT (the size of which is included in the IO data); we impose this as a flat rate on all inputs.
* Set the labor income tax rate at 0.47 (roughly the average for 2018 data).
* We set the level of lump sum transfers at 40\% of labor income (roughly the case for 2018 data).
* Finally, *for now/temporarily*, adjust the value of *TotalTax* to ensure that the consumer has a balanced budget. If we used detailed data on regulation on households, we could drop this assumption.

In [161]:
output = adj.rc_pd(db('vS'), db['s_HH'])
inputs = adj.rc_pd(db('vD'), db['s_HH'])
rTransfer = 0.4004
rLaborTax = 0.478
rVAT = adj.rc_pd(db('vTax').xs('Moms',level='taxTypes'), db['s_HH']) / pyDatabases.pdSum(inputs, 'n')
db.aom(adjMultiIndex.applyMult(rVAT,inputs.index), name = 'tauD') # add flat rate VAT on all demand
db.aom(pd.Series(rLaborTax, index = output.index), name = 'tauS') # add a flat labor tax rate
db.aom(pyDatabases.pdSum(-output * rTransfer, 'n'), name = 'tauLump') # Adjust lump sum tax
db.aom(pyDatabases.pdSum(rVAT * inputs, 'n')+pyDatabases.pdSum(output, 'n') * (rLaborTax-rTransfer), name = 'TotalTax') # adhoc adjustment of 'TotalTax'

Define sector-specific prices given regulation:

In [162]:
db.aom(stdSort(adj.rc_pd((1+db('tauD'))*db('p'), inputs)).add(adj.rc_pd(pW, db('s_HH')), fill_value=0), name = 'pD')

## 8. Government

In [163]:
sIdx = db('s_G')

### 8.1. Nesting structure

Cosnumption aggregate to services, materials, and energy (very similar to investment firm):

In [164]:
mu = pd.MultiIndex.from_tuples([('GC','ZY'), ('GC','ZO'), ('GC','ZE')], names = ['n','nn'])

*Service demands:*

In [165]:
ns = db('n_p').difference(pd.Index(['Energy'],name='n')) # goods included in the "service" branch
mZY = pd.MultiIndex.from_product([['ZY'], 'ZY_'+ns], names = ['n','nn']) # service aggregates
mZYD = pd.MultiIndex.from_arrays([mZY.levels[-1], ns], names = ['n','nn']) # domestic service demand
mZYF = pd.MultiIndex.from_arrays([mZY.levels[-1], ns+'_F'], names = ['n','nn']) # foreign service demand

*Materials nest: Split sequentially into material types, virgin/recycled, domestic/foreign orign:*

In [166]:
mZO = pd.MultiIndex.from_product([['ZO'], 'RxE_'+nmTypes], names = ['n','nn'])
mZOV = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_V'], names = ['n','nn'])
mZOW = pd.MultiIndex.from_arrays([mZO.levels[-1], mZO.levels[-1]+'_R'], names = ['n','nn'])
nmV = mZOV.levels[-1].str.slice(4) # remove RxE_
nmW = mZOW.levels[-1].str.slice(4) # remove RxE_
mZOVD = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV], names = ['n','nn'])
mZOVF = pd.MultiIndex.from_arrays([mZOV.levels[-1], nmV+'_F'], names = ['n','nn'])
mZOWD = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW], names = ['n','nn'])
mZOWF = pd.MultiIndex.from_arrays([mZOW.levels[-1], nmW+'_F'], names = ['n','nn'])

*Energy nest:*

In [167]:
ne = pd.Index(['Energy','WasteE'], name = 'n')
mZE = pd.MultiIndex.from_product([['ZE'], 'ZE_'+ne], names = ['n','nn'])
mZED = pd.MultiIndex.from_arrays([mZE.levels[-1], mZE.levels[-1].str.slice(3)], names = ['n','nn'])
mZEF = pd.MultiIndex.from_arrays([mZE.levels[-1], mZE.levels[-1].str.slice(3)+'_F'], names = ['n','nn'])

Collect in one mapping and add sector index:

In [168]:
mFull = reduce(pd.Index.union, [mu, mZY, mZYD, mZYF, mZO, mZOV, mZOW, mZOVD, mZOVF, mZOWD, mZOWF, mZE, mZED, mZEF])
m = pyDatabases.cartesianProductIndex([sIdx, mFull])

Next, we "trim" the nesting tree - i.e. we remove redundant nests. *NB: This only works with pure "input trees".*

In [169]:
sparsity = adj.rc_pd(db('vD')[db('vD')!=0], sIdx).droplevel('t').index
m = nestingTree.trimNestingStructure(m, sparsity)

Save nesting tree:

In [170]:
db['nestG'] = m

### 8.2. Elasticities:

*CES upper nest:*

In [171]:
sigmaC = pd.Series(0.5, index = m[m.get_level_values('n') == 'GC'].droplevel('nn').unique(), name = 'sigma')

*CES, materials:*

In [172]:
sigmaZO = pd.Series(0.25, index = m[m.get_level_values('n')=='ZO'].droplevel('nn').unique(), name = 'sigma')
sigmaZOnm = pd.Series(3, index = m[m.get_level_values('n').isin('RxE_'+nmTypes)].droplevel('nn').unique(), name = 'sigma')
sigmaZOVW = pd.Series(3, index = m[m.get_level_values('n').isin(('RxE_'+nmTypes+'_V').union('RxE_'+nmTypes+'_R'))].droplevel('nn').unique(), name = 'sigma')

*CES elasticities, services:*

In [173]:
sigmaZY = pd.Series(0.25, index = m[m.get_level_values('n') == 'ZY'].droplevel('nn').unique(), name = 'sigma')
sigmaZYym = pd.Series(3, index = m[m.get_level_values('n').isin('ZY_'+ns)].droplevel('nn').unique(), name = 'sigma')

*CES elasticities, energy:*

In [174]:
sigmaZE = pd.Series(0.25, index = m[m.get_level_values('n') == 'ZE'].droplevel('nn').unique(), name = 'sigma')
sigmaZEym = pd.Series(3, index = m[m.get_level_values('n').isin('ZE_'+ne)].droplevel('nn').unique(), name = 'sigma')

Add to database:

In [175]:
sigma = pd.concat([sigmaC, sigmaZO, sigmaZOnm, sigmaZOVW, sigmaZY, sigmaZYym, sigmaZE, sigmaZEym], axis =0)
db.aom(sigma, name = 'sigma') # elasticity of substitution

### 8.3. Regulation

The government regulates/taxes itself (because data says so). Because we use incomplete data on taxes (especially on household taxes/transfers), we make some adhoc adjustments here:
* Remove taxes on inventory (because the "inventory sector" is only used for completeness of the IO system).
* Set flat VAT tax rate on government consumption to target taxes from government sector.

*Note: Currently, we do not have waste generation in the government sector.*

In [176]:
inputs = adj.rc_pd(db('vD'), db('s_G'))
db['TotalTax'] = adj.rc_pd(db('TotalTax'), ('not', pd.Index(['itory'],name='s'))) # remove inventory taxes
db['d_TotalTax'] = db['TotalTax'].index.droplevel('t').unique() # what sectors pay taxes
rVAT = adj.rc_pd(db('TotalTax'), db('s_G')) / pyDatabases.pdSum(inputs,'n') # define VAT rate of taxes
db.aom(adjMultiIndex.applyMult(rVAT,inputs.index), name = 'tauD') # add flat rate VAT on all demand
db.aom(stdSort(adj.rc_pd((1+db('tauD'))*db('p'), inputs)), name = 'pD')

## 9. Trade

We apply a simple Armington specification for all exports. This means that the demand is a simple isoelastic function of the relative price of domestic/foreign goods.

In [177]:
db.aom(pd.Series(6, index = db('dExport')), name='sigma')

With export of waste management services being this isoelastic function, the implicit assumption is that import of waste materials is a similar iso-elastic function. This

*Note: We should add some information on regulation on trade at some point. Here, just add flat tariff/subsidy on all exports:* 

In [178]:
avgTariffRate = adj.rc_pd(db('TotalTax'), db('s_f'))/pyDatabases.pdSum(adj.rc_pd(db('vD'), db('s_f')), 'n')
TariffRate = stdSort(adjMultiIndex.applyMult(avgTariffRate, db('dExport')))
db.aom(TariffRate, name = 'tauD')
db.aom(pd.Series(0, index = adj.rc_pd(db('TotalTax'), db('s_f')).index), name = 'tauLump')
db.aom(stdSort(adj.rc_pd((1+db('tauD'))*db('p'), db('s_f'))), name = 'pD')

## 10. Emissions

Set taxes to zero for emissions:

In [179]:
db['uCO2'] = adj.rc_pd(db('qCO2'), db('s_p')) / adj.rc_pd(db('qS'), db('s_p'))
db['tauCO2agg'] = pd.Series(0, index = db('t0'))
db['tauDist'] = pd.Series(1, index = db('uCO2').index)
db['tauCO2'] = pd.Series(0, index = db('uCO2').index)
db['qCO2agg'] = db('qCO2').groupby('t').sum()

For now, define the effective tax rate used in production modules as the copy of true tax:

In [180]:
db['tauEffCO2'] = db('tauCO2').copy()

## 11. Globals

Other parameters/variables used throughout:

In [181]:
db.aom(pd.Series(db('R_LR'), index = db('t')), name = 'Rrate', priority = 'first') # fix interest rate path at long run level

## Export

Clean up symbols:

In [182]:
removeSyms = ('taxTypes','vTax', 'vWE', 'vWE_F')
[db.series.__delitem__(k) for k in removeSyms if k in db.symbols];

Export:

In [183]:
db.export(repo=d['data'])